# 🧪 Thực Nghiệm Độc Lập: Bộ 3 Bài Test Chứng Minh Điểm Yếu Của LiDAR Trên Google Colab
### Khung Lý Thuyết: Dimension-Free Lipschitz Bound & Randomized Smoothing (RS-LiDAR)

Notebook này là môi trường độc lập thực thi **Bộ 3 Bài Test Chuẩn Xác Theo Khung Lý Thuyết**:
1. **Bài Test 1 (Solver Error Robustness & Khảo sát $\sigma$)**: So sánh quỹ đạo không dẫn đường giữa DPM-5 (lookahead) và DDIM-50 (chuẩn), đo sai số reward trên đa thang đo (ImageReward, CLIP, HPSv2) và khảo sát $\sigma \in \{0.15, 0.30, 0.60\}$.
2. **Bài Test 2 (Softmax Mode Collapse / Entropy)**: Đo độ sụp đổ Entropy Shannon $H(w^r)$, chứng minh hiện tượng Best-of-1 Trap.
3. **Bài Test 3 (Guidance Vector Field Stability)**: Đo độ ổn định góc quay Cosine $\text{CosSim}(\mathbf{g}_t, \mathbf{g}_{t+\delta})$ khi có nhiễu vi mô $\delta = 10^{-3}$.

*Lưu ý*: Test 2 và Test 3 hoàn toàn là phép tính ma trận đại số trên latent, chạy chỉ mất ~10 giây!

## 1. Kiểm Tra Phần Cứng GPU

In [ ]:
import torch, sys
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA khả dụng:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM: {round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)} GB")
else:
    raise RuntimeError("❌ Vui lòng chọn Runtime -> Change runtime type -> GPU!")
!nvidia-smi

## 2. Gắn Kết Google Drive & Thiết Lập Mã Nguồn

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')
DRIVE_DIR = "/content/drive/MyDrive/RS-LiDAR/test_results"
os.makedirs(DRIVE_DIR, exist_ok=True)

REPO_DIR = "/content/RS-LiDAR"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/leekwanreal/RS-LiDAR.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull origin main

WORKDIR = REPO_DIR if os.path.exists(f"{REPO_DIR}/test_lidar_weaknesses.py") else f"{REPO_DIR}/Diffusion-LiDAR-Sampling"
%cd {WORKDIR}
print("✅ Sẵn sàng tại:", WORKDIR)

## 3. Cài Đặt Môi Trường Đầy Đủ

In [ ]:
import os, urllib.request

os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"

!pip install -q --upgrade protobuf
!pip install -q transformers==4.38.2 diffusers==0.31.0 accelerate==1.2.1 safetensors huggingface-hub einops ftfy timm peft
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q git+https://github.com/THUDM/ImageReward.git
!pip install -q hpsv2 matplotlib tqdm scipy seaborn pandas tabulate

import hpsv2
hpsv2_vocab = os.path.join(os.path.dirname(hpsv2.__file__), "src", "open_clip", "bpe_simple_vocab_16e6.txt.gz")
os.makedirs(os.path.dirname(hpsv2_vocab), exist_ok=True)
if not os.path.exists(hpsv2_vocab):
    urllib.request.urlretrieve("https://github.com/openai/CLIP/raw/main/clip/bpe_simple_vocab_16e6.txt.gz", hpsv2_vocab)
print("✅ Môi trường cho Bộ 3 Bài Test đã hoàn tất!")

## 4. Chạy Toàn Bộ 3 Bài Test Thực Nghiệm (`test_lidar_weaknesses.py`)
Cấu hình chuẩn khoa học: 20 prompts, 20 particles, khảo sát sigma $\sigma \in \{0.15, 0.30, 0.60\}$. Toàn bộ biểu đồ và bảng số liệu sẽ tự động lưu ra Google Drive.

In [ ]:
!python test_lidar_weaknesses.py \
    --test all \
    --num_prompts 20 \
    --num_particles 20 \
    --tune_sigma \
    --sigmas "0.15,0.30,0.60" \
    --output_dir "{DRIVE_DIR}"

## 5. Hiển Thị Bảng Kết Quả & Đồ Thị So Sánh Trực Quan

In [ ]:
import pandas as pd, glob, os
from IPython.display import display, Image

# 1. Hiển thị Bảng so sánh 3 bài test
table_files = glob.glob(f"{DRIVE_DIR}/*comparison*.csv")
if table_files:
    df = pd.read_csv(table_files[0])
    print("📊 BẢNG KẾT QUẢ SO SÁNH 3 BÀI TEST:")
    display(df)

# 2. Hiển thị Bảng khảo sát Sigma
sigma_files = glob.glob(f"{DRIVE_DIR}/*sigma*.csv")
if sigma_files:
    df_sigma = pd.read_csv(sigma_files[0])
    print("\n📈 BẢNG KHẢO SÁT THAM SỐ SIGMA (ABLATION STUDY):")
    display(df_sigma)

# 3. Hiển thị các đồ thị biểu diễn
png_files = sorted(glob.glob(f"{DRIVE_DIR}/*.png"))
for p in png_files:
    print(f"\n🖼️ Đồ thị: {os.path.basename(p)}")
    display(Image(filename=p))